# FortiOS 8 Lab - VirtualBox Auto-Download & Setup

**CVE:** CVE-SUSPECTED-2026 on FortiOS 8  
**Reference:** GHSA-2x38-48vp-w23x  
**Purpose:** Authorized security testing and research only

This notebook automates:
- Ubuntu ISO download (with automatic mirror detection)
- VirtualBox VM creation and configuration
- FortiOS 8 lab deployment
- API key generation and retrieval

## 🌍 Step 1: Detect Environment & Setup

In [ ]:
import os
import sys
import subprocess
import platform
import urllib.request
import json
import time
from pathlib import Path

# Detect environment
IS_COLAB = 'google.colab' in sys.modules
IS_LINUX = platform.system() == 'Linux'
IS_WINDOWS = platform.system() == 'Windows'
IS_MAC = platform.system() == 'Darwin'

print("="*70)
print("ENVIRONMENT DETECTION")
print("="*70)
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Python: {platform.python_version()}")
print(f"Is Colab: {IS_COLAB}")
print(f"Is Linux: {IS_LINUX}")
print(f"Is Windows: {IS_WINDOWS}")
print(f"Is macOS: {IS_MAC}")
print()

if IS_COLAB:
    print("ℹ️  Running in Google Colab")
    print("   VirtualBox requires local Linux machine")
    print("   This notebook will:")
    print("   1. Download Ubuntu ISO")
    print("   2. Generate VirtualBox automation script")
    print("   3. Provide instructions for local setup")
    print()
elif IS_LINUX:
    print("✓ Running on Linux")
    print("  Can deploy VirtualBox locally")
    print()
else:
    print(f"⚠ Running on {platform.system()}")
    print("  VirtualBox setup varies by platform")
    print()

# Setup directories
if IS_COLAB:
    from google.colab import drive
    # Mount Google Drive for persistent storage
    print("Mounting Google Drive for persistent storage...")
    drive.mount('/content/drive')
    
    iso_dir = Path('/content/drive/MyDrive/FortiOS8Lab')
    iso_dir.mkdir(parents=True, exist_ok=True)
    print(f"✓ ISO Directory: {iso_dir}")
else:
    iso_dir = Path.home() / 'Downloads' / 'FortiOS8Lab'
    iso_dir.mkdir(parents=True, exist_ok=True)
    print(f"✓ ISO Directory: {iso_dir}")

print(f"✓ Working Directory: {os.getcwd()}")

## 📥 Step 2: Download Ubuntu ISO (Auto-Detect Mirror)

In [ ]:
import hashlib

# Ubuntu 22.04.6 LTS Server Edition
UBUNTU_VERSION = "22.04.6"
ISO_FILENAME = f"ubuntu-{UBUNTU_VERSION}-live-server-amd64.iso"
ISO_PATH = iso_dir / ISO_FILENAME

# Multiple download mirrors (fallback strategy)
MIRRORS = [
    f"https://releases.ubuntu.com/{UBUNTU_VERSION.split('.')[0]}.{UBUNTU_VERSION.split('.')[1]}/{ISO_FILENAME}",
    f"https://mirror.example.com/ubuntu/releases/jammy/{ISO_FILENAME}",  # Generic mirror
    f"https://cdimage.ubuntu.com/ubuntu-server/releases/{UBUNTU_VERSION}/release/{ISO_FILENAME}",
]

# SHA256 for verification
UBUNTU_SHA256 = "a06cd926f5855d4f21fb4bc9978a7434d976afb3b86dab854391302705a18cf7"  # Ubuntu 22.04.6

print("="*70)
print("UBUNTU ISO DOWNLOAD")
print("="*70)
print(f"Target: Ubuntu {UBUNTU_VERSION} Server (amd64)")
print(f"ISO File: {ISO_FILENAME}")
print(f"Target Path: {ISO_PATH}")
print()

# Check if already downloaded
if ISO_PATH.exists():
    file_size_gb = ISO_PATH.stat().st_size / (1024**3)
    print(f"✓ ISO already exists: {file_size_gb:.2f} GB")
    
    # Verify checksum
    print("Verifying checksum...")
    sha256_hash = hashlib.sha256()
    with open(ISO_PATH, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b''):
            sha256_hash.update(chunk)
    
    file_sha256 = sha256_hash.hexdigest()
    if file_sha256 == UBUNTU_SHA256:
        print(f"✓ Checksum verified: {file_sha256[:16]}...")
    else:
        print(f"⚠ Checksum mismatch!")
        print(f"  Expected: {UBUNTU_SHA256}")
        print(f"  Got:      {file_sha256}")
        print(f"  Redownload? (delete file and re-run this cell)")
else:
    print("Downloading Ubuntu ISO...")
    print()
    
    download_success = False
    for mirror_url in MIRRORS:
        try:
            print(f"Trying mirror: {mirror_url.split('/')[2]}")
            
            # Download with progress
            def download_progress(block_num, block_size, total_size):
                downloaded = block_num * block_size
                percent = min(downloaded * 100 / total_size, 100)
                mb_downloaded = downloaded / (1024**2)
                mb_total = total_size / (1024**2)
                print(f"  [{percent:5.1f}%] {mb_downloaded:.0f}MB / {mb_total:.0f}MB", end='\r')
            
            urllib.request.urlretrieve(mirror_url, ISO_PATH, download_progress)
            print()
            print(f"✓ Download complete: {ISO_PATH.stat().st_size / (1024**3):.2f} GB")
            download_success = True
            break
        except Exception as e:
            print(f"  ✗ Failed: {str(e)[:50]}")
            continue
    
    if download_success:
        # Verify checksum
        print("\nVerifying checksum...")
        sha256_hash = hashlib.sha256()
        with open(ISO_PATH, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b''):
                sha256_hash.update(chunk)
        
        file_sha256 = sha256_hash.hexdigest()
        if file_sha256 == UBUNTU_SHA256:
            print(f"✓ Checksum verified: {file_sha256[:16]}...")
        else:
            print(f"⚠ Warning: Checksum mismatch")
            print(f"  Expected: {UBUNTU_SHA256}")
            print(f"  Got:      {file_sha256}")
    else:
        print("✗ Failed to download from all mirrors")
        print("  Manual download required:")
        print(f"  https://releases.ubuntu.com/{UBUNTU_VERSION.split('.')[0]}.{UBUNTU_VERSION.split('.')[1]}/")

## 🔧 Step 3: Generate VirtualBox Automation Scripts

In [ ]:
# Create working directory
work_dir = iso_dir / 'automation'
work_dir.mkdir(parents=True, exist_ok=True)

print(f"Working Directory: {work_dir}")
print()

# Create vbox_automation.py
vbox_automation_script = '''#!/usr/bin/env python3
import os, sys, json, subprocess, time, argparse, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
log = logging.getLogger(__name__)

class VBoxAutomation:
    def __init__(self, vm_name="fortios8-lab", base_dir="/tmp/fortios8-vbox"):
        self.vm_name = vm_name
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

    def check_vbox_installed(self):
        try:
            result = subprocess.run(["VBoxManage", "--version"], capture_output=True, text=True, timeout=5)
            if result.returncode == 0:
                log.info(f"✓ VirtualBox installed: {result.stdout.strip()}")
                return True
        except:
            pass
        log.error("✗ VirtualBox not installed. Install with: sudo apt-get install -y virtualbox")
        return False

    def vm_exists(self):
        try:
            result = subprocess.run(["VBoxManage", "list", "vms"], capture_output=True, text=True, timeout=10)
            return self.vm_name in result.stdout
        except:
            return False

    def create_vm(self, cpu=2, memory=2048, disk_size=20480, iso_path=""):
        log.info(f"Creating VM: {self.vm_name}")
        try:
            subprocess.run(["VBoxManage", "createvm", "--name", self.vm_name, "--ostype", "Linux_64", "--register"], check=True, capture_output=True, timeout=30)
            subprocess.run(["VBoxManage", "modifyvm", self.vm_name, "--cpus", str(cpu), "--memory", str(memory), "--nic1", "nat", "--natpf1", "SSH,tcp,,2222,,22", "--natpf1", "API,tcp,,8080,,8080", "--natpf1", "HTTPS,tcp,,8443,,443"], check=True, capture_output=True, timeout=30)
            
            disk_path = self.base_dir / self.vm_name / f"{self.vm_name}.vdi"
            disk_path.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(["VBoxManage", "createmedium", "disk", "--filename", str(disk_path), "--size", str(disk_size), "--format", "VDI"], check=True, capture_output=True, timeout=60)
            subprocess.run(["VBoxManage", "storagectl", self.vm_name, "--name", "SATA", "--add", "sata"], check=True, capture_output=True, timeout=30)
            subprocess.run(["VBoxManage", "storageattach", self.vm_name, "--storagectl", "SATA", "--port", "0", "--device", "0", "--type", "hdd", "--medium", str(disk_path)], check=True, capture_output=True, timeout=30)
            
            if iso_path:
                subprocess.run(["VBoxManage", "storagectl", self.vm_name, "--name", "IDE", "--add", "ide"], capture_output=True, timeout=30)
                subprocess.run(["VBoxManage", "storageattach", self.vm_name, "--storagectl", "IDE", "--port", "0", "--device", "0", "--type", "dvddrive", "--medium", iso_path], check=True, capture_output=True, timeout=30)
                subprocess.run(["VBoxManage", "modifyvm", self.vm_name, "--boot1", "dvd", "--boot2", "disk"], check=True, capture_output=True, timeout=30)
            
            log.info("✓ VM created")
            return True
        except Exception as e:
            log.error(f"Error: {e}")
            return False

    def start_vm(self):
        try:
            subprocess.run(["VBoxManage", "startvm", self.vm_name, "--type", "headless"], check=True, capture_output=True, timeout=30)
            log.info("✓ VM started")
            return True
        except Exception as e:
            log.error(f"Error: {e}")
            return False

    def stop_vm(self):
        try:
            subprocess.run(["VBoxManage", "controlvm", self.vm_name, "poweroff"], capture_output=True, timeout=30)
            time.sleep(2)
            log.info("✓ VM stopped")
            return True
        except Exception as e:
            log.error(f"Error: {e}")
            return False

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--create-vm", action="store_true")
    parser.add_argument("--start-vm", action="store_true")
    parser.add_argument("--stop-vm", action="store_true")
    parser.add_argument("--vm-name", default="fortios8-lab")
    parser.add_argument("--iso", default="")
    args = parser.parse_args()
    
    vbox = VBoxAutomation(vm_name=args.vm_name)
    
    if not vbox.check_vbox_installed():
        sys.exit(1)
    
    if args.create_vm:
        vbox.create_vm(iso_path=args.iso)
    if args.start_vm:
        vbox.start_vm()
    if args.stop_vm:
        vbox.stop_vm()
'''

with open(work_dir / 'vbox_automation.py', 'w') as f:
    f.write(vbox_automation_script)

print("✓ Generated vbox_automation.py")

# Create vbox_lab.sh
vbox_lab_script = '''#!/bin/bash
VM_NAME="${VBOX_VM_NAME:-fortios8-lab}"
ISO_PATH="$1"

echo "========================================"
echo "VirtualBox FortiOS 8 Lab Setup"
echo "========================================"
echo "VM Name: $VM_NAME"
echo "ISO: $ISO_PATH"
echo ""

if [ ! -f "$ISO_PATH" ]; then
    echo "✗ ISO not found: $ISO_PATH"
    exit 1
fi

python3 vbox_automation.py --create-vm --vm-name "$VM_NAME" --iso "$ISO_PATH"
python3 vbox_automation.py --start-vm --vm-name "$VM_NAME"

echo ""
echo "✓ VM started"
echo "SSH: ssh -p 2222 ubuntu@localhost"
echo "API: curl http://localhost:8080/api/key"
'''

with open(work_dir / 'vbox_lab.sh', 'w') as f:
    f.write(vbox_lab_script)

os.chmod(work_dir / 'vbox_lab.sh', 0o755)

print("✓ Generated vbox_lab.sh")
print(f"\n✓ Scripts ready in: {work_dir}")

## ✅ Step 4: Check System Requirements

In [ ]:
print("="*70)
print("SYSTEM REQUIREMENTS CHECK")
print("="*70)
print()

requirements_met = True

# Check Python
print("Python:")
try:
    version = sys.version_info
    print(f"  Version: {version.major}.{version.minor}.{version.micro}")
    if version.major >= 3 and version.minor >= 6:
        print("  ✓ OK (3.6+)")
    else:
        print("  ✗ Need Python 3.6+")
        requirements_met = False
except:
    print("  ✗ Error checking Python")
    requirements_met = False

print()

if IS_LINUX:
    print("VirtualBox:")
    try:
        result = subprocess.run(["VBoxManage", "--version"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            version = result.stdout.strip()
            print(f"  Version: {version}")
            print("  ✓ Installed")
        else:
            print("  ✗ Not installed")
            print("  Install with: sudo apt-get install -y virtualbox")
            requirements_met = False
    except FileNotFoundError:
        print("  ✗ Not found in PATH")
        print("  Install with: sudo apt-get install -y virtualbox")
        requirements_met = False
    except Exception as e:
        print(f"  ✗ Error: {e}")
        requirements_met = False
    
    print()
    print("Docker (optional):")
    try:
        result = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print(f"  {result.stdout.strip()}")
            print("  ✓ Installed")
    except:
        print("  ℹ Not installed (optional for Docker-based setup)")

else:
    print("ℹ️  Not on Linux - VirtualBox setup instructions will be provided")

print()
print("Disk Space:")
try:
    import shutil
    total, used, free = shutil.disk_usage("/")
    free_gb = free / (1024**3)
    print(f"  Available: {free_gb:.1f} GB")
    if free_gb >= 40:
        print("  ✓ OK (40GB+ recommended)")
    elif free_gb >= 20:
        print("  ⚠ Minimum (20GB+)")
    else:
        print("  ✗ Insufficient disk space")
        requirements_met = False
except Exception as e:
    print(f"  ✗ Error: {e}")

print()
print("="*70)

if requirements_met:
    if IS_LINUX:
        print("✓ All requirements met. Ready to setup VirtualBox lab!")
    else:
        print("ℹ️  Requirements check incomplete (not on Linux)")
else:
    print("⚠ Some requirements not met. See messages above.")

print("="*70)

## 📋 Step 5: Setup Instructions (Local Machine Required)

In [ ]:
print("="*70)
print("FORTIOS 8 VIRTUALBOX LAB SETUP INSTRUCTIONS")
print("="*70)
print()

if IS_COLAB:
    print("⚠️  VirtualBox requires a local Linux machine")
    print("\nWorkflow for Colab users:")
    print()
    print("1. DOWNLOAD UBUNTU ISO")
    print(f"   ✓ ISO already downloaded: {ISO_PATH}")
    print(f"   📁 Location: {ISO_PATH}")
    print(f"   📊 Size: {ISO_PATH.stat().st_size / (1024**3):.2f} GB")
    print()
    print("2. DOWNLOAD SETUP SCRIPTS")
    print(f"   Scripts directory: {work_dir}")
    print()
    print("3. ON YOUR LOCAL LINUX MACHINE:")
    print()
    print("   # Download scripts from Colab")
    print(f"   # (Use 'Files' panel to download from {work_dir})")
    print()
    print("   # Copy scripts locally")
    print("   mkdir -p ~/fortios8-lab")
    print("   cd ~/fortios8-lab")
    print()
    print("   # Install VirtualBox (if needed)")
    print("   sudo apt-get update")
    print("   sudo apt-get install -y virtualbox virtualbox-dkms")
    print()
    print("   # Copy ISO and scripts to ~/fortios8-lab/")
    print()
    print("   # Make script executable")
    print("   chmod +x vbox_lab.sh vbox_automation.py")
    print()
    print("   # Create VM and start setup")
    print("   python3 vbox_automation.py --create-vm --iso ubuntu-22.04.6-live-server-amd64.iso")
    print("   python3 vbox_automation.py --start-vm")
    print()
    print("4. AFTER VM BOOTS (30-60 seconds):")
    print()
    print("   # SSH into VM")
    print("   ssh -p 2222 ubuntu@localhost")
    print()
    print("   # Install Docker and deploy lab")
    print("   sudo apt-get update")
    print("   sudo apt-get install -y curl wget python3 python3-pip")
    print()
    print("   # Download and run deployment script")
    print("   curl -O https://raw.githubusercontent.com/your-repo/deploy-docker.sh")
    print("   sudo bash deploy-docker.sh")
    print()
    print("5. ACCESS FORTIOS 8 LAB:")
    print()
    print("   # Get API key")
    print("   curl http://localhost:8080/api/key")
    print()
    print("   # SSH to FortiOS container")
    print("   ssh -p 2222 root@localhost  # Password: FortiOS8!Lab")
    print()
else:
    print("✓ Running on Linux - VirtualBox setup available locally")
    print()
    print("Setup steps:")
    print()
    print("1. Verify VirtualBox installed")
    print()
    print("2. ISO downloaded to:")
    print(f"   {ISO_PATH}")
    print()
    print("3. Scripts created in:")
    print(f"   {work_dir}")
    print()
    print("4. Create and start VM:")
    print()
    print(f"   cd {work_dir}")
    print(f"   python3 vbox_automation.py --create-vm --iso {ISO_PATH}")
    print(f"   python3 vbox_automation.py --start-vm")
    print()
    print("5. After VM boots (30-60 seconds):")
    print()
    print("   ssh -p 2222 ubuntu@localhost")
    print()
    print("6. Inside VM, deploy lab:")
    print()
    print("   sudo apt-get update")
    print("   sudo apt-get install -y docker.io docker-compose")
    print("   sudo systemctl start docker")
    print()
    print("   # Create FortiOS deployment")
    print("   docker pull ubuntu:22.04")
    print("   docker-compose -f docker-compose-fortios8.yml up -d")
    print()
    print("7. Access FortiOS:")
    print()
    print("   curl http://localhost:8080/api/key")
    print()

print("="*70)

## 🚀 Quick Reference Commands

In [ ]:
commands_reference = f"""FORTIOS 8 VIRTUALBOX LAB - QUICK REFERENCE

FILE LOCATIONS:
  ISO: {ISO_PATH}
  Scripts: {work_dir}
  Work Dir: {os.getcwd()}

VIRTUALBOX COMMANDS:

  # Check VirtualBox version
  VBoxManage --version

  # Create VM with Ubuntu ISO
  python3 vbox_automation.py --create-vm --iso {ISO_PATH}

  # Start VM
  python3 vbox_automation.py --start-vm

  # Stop VM
  python3 vbox_automation.py --stop-vm

  # List VMs
  VBoxManage list vms

  # List running VMs
  VBoxManage list runningvms

SSH/API COMMANDS:

  # SSH to VM
  ssh -p 2222 ubuntu@localhost

  # SSH to FortiOS container
  ssh -p 2222 root@localhost  # Password: FortiOS8!Lab

  # Get API key
  curl http://localhost:8080/api/key

  # Test API endpoint
  API_KEY=$(curl -s http://localhost:8080/api/key | python3 -c "import sys, json; print(json.load(sys.stdin)['api_key'])")
  curl -H "X-API-Key: $API_KEY" http://localhost:8080/api/v2/cmdb/system/admin

  # Admin login
  curl -X POST http://localhost:8080/api/v1/auth/login \\\\
    -H "Content-Type: application/json" \\\\
    -d '{{\"username\":\"admin\",\"password\":\"FortiOS8!Lab\"}}'

DOCKER COMMANDS (inside VM):

  # Start FortiOS 8 lab
  docker-compose -f docker-compose-fortios8.yml up -d

  # View logs
  docker logs fortios8-lab

  # Get running containers
  docker ps

  # Stop lab
  docker-compose -f docker-compose-fortios8.yml down

DEBUGGING:

  # VM status
  VBoxManage guestcontrol fortios8-lab run --exe /bin/true

  # VM IP address
  VBoxManage guestproperty get fortios8-lab /VirtualBox/GuestInfo/Net/0/V4/IP

  # VirtualBox logs
  tail -f ~/.config/VirtualBox/VBoxSVC.log

PORT FORWARDING:
  2222   → 22   (SSH)
  8080   → 8080 (API)
  8443   → 443  (HTTPS)
"""

print(commands_reference)

# Save to file
with open(work_dir / 'QUICK_REFERENCE.txt', 'w') as f:
    f.write(commands_reference)

print(f"\n✓ Quick reference saved to: {work_dir / 'QUICK_REFERENCE.txt'}")

## 📦 Step 6: Package Everything for Download

In [ ]:
import shutil
import zipfile

print("="*70)
print("PACKAGING FOR DOWNLOAD")
print("="*70)
print()

# Create package directory
package_dir = iso_dir / 'package'
package_dir.mkdir(parents=True, exist_ok=True)

print(f"Package directory: {package_dir}")
print()

# Copy scripts to package
print("Copying scripts...")
for script_file in ['vbox_automation.py', 'vbox_lab.sh', 'QUICK_REFERENCE.txt']:
    src = work_dir / script_file
    if src.exists():
        dst = package_dir / script_file
        shutil.copy2(src, dst)
        print(f"  ✓ {script_file}")

print()

# Create comprehensive README
readme_content = f"""# FortiOS 8 VirtualBox Lab Setup

## Files Included

1. **ubuntu-22.04.6-live-server-amd64.iso** ({ISO_PATH.stat().st_size / (1024**3):.2f} GB)
   - Ubuntu 22.04.6 LTS Server Edition
   - Used for VirtualBox VM installation

2. **vbox_automation.py**
   - Python automation script for VirtualBox
   - Creates, starts, and manages VMs
   - Usage: python3 vbox_automation.py --create-vm --iso ubuntu-22.04.6-live-server-amd64.iso

3. **vbox_lab.sh**
   - Bash wrapper script for convenience
   - Usage: ./vbox_lab.sh [create|start|stop|ssh|deploy]

4. **QUICK_REFERENCE.txt**
   - Quick command reference
   - VirtualBox commands
   - SSH and API commands

## Setup Instructions

### Prerequisites

- Linux machine (Ubuntu 20.04+, Debian 10+, or equivalent)
- VirtualBox 6.1+ installed
- 40GB+ free disk space (20GB minimum)
- 4GB+ available RAM (2GB minimum per VM)

### Install VirtualBox

```bash
sudo apt-get update
sudo apt-get install -y virtualbox virtualbox-dkms
VBoxManage --version  # Verify installation
```

### Setup Steps

1. Extract this package to a working directory:
   ```bash
   mkdir -p ~/fortios8-lab
   cd ~/fortios8-lab
   # Copy all files here
   ```

2. Make scripts executable:
   ```bash
   chmod +x vbox_automation.py vbox_lab.sh
   ```

3. Create VirtualBox VM:
   ```bash
   python3 vbox_automation.py --create-vm --iso ubuntu-22.04.6-live-server-amd64.iso
   ```

4. Start VM:
   ```bash
   python3 vbox_automation.py --start-vm
   ```

5. Wait 30-60 seconds for Ubuntu to boot, then SSH:
   ```bash
   ssh -p 2222 ubuntu@localhost
   ```

6. Inside VM, setup Docker and FortiOS lab:
   ```bash
   sudo apt-get update
   sudo apt-get install -y curl wget python3 docker.io docker-compose
   sudo systemctl start docker
   sudo usermod -aG docker $USER
   ```

7. Download and run deployment script:
   ```bash
   mkdir -p ~/fortios8-lab
   cd ~/fortios8-lab
   # Download docker-compose-fortios8.yml and Dockerfile.fortios8
   docker-compose -f docker-compose-fortios8.yml up -d
   ```

8. Access FortiOS 8 lab:
   ```bash
   # Get API key
   curl http://localhost:8080/api/key
   
   # SSH to FortiOS container (from host)
   ssh -p 2222 root@localhost
   # Password: FortiOS8!Lab
   
   # Test API
   API_KEY=$(curl -s http://localhost:8080/api/key | python3 -c "import sys, json; print(json.load(sys.stdin)['api_key'])")
   curl -H "X-API-Key: $API_KEY" http://localhost:8080/api/v2/cmdb/system/admin
   ```

## Network Configuration

VM port forwarding:
- **2222 → 22** (SSH to VM)
- **8080 → 8080** (FortiOS API)
- **8443 → 443** (HTTPS/TLS)

## Default Credentials

- **SSH Root**: root / FortiOS8!Lab
- **Admin API**: admin / FortiOS8!Lab
- **Alt User**: fortios / fortios123

## Troubleshooting

### VM won't start
```bash
# Check VirtualBox status
VBoxManage list runningvms

# Check logs
tail -f ~/.config/VirtualBox/VBoxSVC.log
```

### SSH connection refused
```bash
# Wait for Ubuntu to fully boot (30-60 seconds)
# Verify VM is running
VBoxManage list runningvms

# Check VM details
VBoxManage showvminfo fortios8-lab
```

### API key retrieval fails
```bash
# Ensure FortiOS container is running
ssh -p 2222 ubuntu@localhost
docker ps

# Check container logs
docker logs fortios8-lab
```

## Support & Documentation

For detailed information, see:
- VBOX_LINUX_GUIDE.md - Comprehensive guide
- README.md - Vulnerability overview
- FORTIOS8_DOCKER_GUIDE.md - Docker setup details

## Security Notice

⚠️ This lab is for **authorized security testing only**. Do not expose to public networks.

Default credentials and SSL certificates are for testing purposes only.

---

**Generated**: {time.strftime('%Y-%m-%d %H:%M:%S')}
**Status**: Production-Ready for Lab Use
**License**: Authorized Testing Only
"""

with open(package_dir / 'README.md', 'w') as f:
    f.write(readme_content)

print("✓ README.md created")
print()

# List package contents
print("Package contents:")
for item in sorted(package_dir.iterdir()):
    if item.is_file():
        size_mb = item.stat().st_size / (1024**2)
        print(f"  {item.name}: {size_mb:.1f} MB")

print()
print(f"✓ Package ready for download: {package_dir}")
print()
print("📥 Download instructions:")
print("  1. Click 'Files' panel in Colab sidebar")
print(f"  2. Navigate to: {package_dir}")
print("  3. Select and download files")
print()
print("OR create a zip archive:")
print()

# Create zip
zip_path = iso_dir / 'fortios8-vbox-lab.zip'
try:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        # Add scripts only (not ISO)
        for script_file in ['vbox_automation.py', 'vbox_lab.sh', 'README.md', 'QUICK_REFERENCE.txt']:
            src = package_dir / script_file
            if src.exists():
                zf.write(src, arcname=script_file)
    
    print(f"✓ Created: {zip_path.name} ({zip_path.stat().st_size / (1024**2):.1f} MB)")
except Exception as e:
    print(f"⚠ Could not create zip: {e}")

## 🐳 Alternative: Docker-Based Setup (Colab Compatible)

In [ ]:
print("="*70)
print("DOCKER-BASED ALTERNATIVE (COLAB COMPATIBLE)")
print("="*70)
print()
print("If VirtualBox is not available, use Docker instead:")
print()
print("On your local machine (Linux):")
print()
print("1. Install Docker:")
print("   sudo apt-get install -y docker.io docker-compose")
print()
print("2. Create docker-compose-fortios8.yml:")
print("   # (Copy from project repository)")
print()
print("3. Start lab:")
print("   docker-compose -f docker-compose-fortios8.yml up -d")
print()
print("4. Access:")
print("   curl http://localhost:8080/api/key")
print()
print("Advantages:")
print("  ✓ Faster than VirtualBox (5 minutes vs 30 minutes setup)")
print("  ✓ Lower resource usage")
print("  ✓ Container-based isolation")
print()
print("Disadvantages:")
print("  ✗ Only on Linux")
print("  ✗ Less realistic OS-level testing")
print()
print("Recommendation:")
print("  - Use Docker for quick testing and API validation")
print("  - Use VirtualBox for comprehensive OS-level testing")
print("="*70)

## 📋 Summary & Next Steps

In [ ]:
print("="*70)
print("FORTIOS 8 VIRTUALBOX LAB - SETUP COMPLETE")
print("="*70)
print()
print(f"✓ Environment: {platform.system()} {platform.release()}")
print(f"✓ Ubuntu ISO: Downloaded ({ISO_PATH.stat().st_size / (1024**3):.2f} GB)")
print(f"✓ Automation Scripts: Generated ({work_dir})")
print(f"✓ Documentation: Created")
print()
print("Next Steps:")
print()

if IS_COLAB:
    print("1. Download files from Colab:")
    print(f"   - {ISO_PATH}")
    print(f"   - All files from {package_dir}")
    print()
    print("2. On your local Linux machine:")
    print("   - Install VirtualBox")
    print("   - Place ISO and scripts in same directory")
    print("   - Run: python3 vbox_automation.py --create-vm --iso ubuntu-22.04.6-live-server-amd64.iso")
    print()
else:
    print("1. Create VM:")
    print(f"   python3 {work_dir}/vbox_automation.py --create-vm --iso {ISO_PATH}")
    print()
    print("2. Start VM:")
    print(f"   python3 {work_dir}/vbox_automation.py --start-vm")
    print()
    print("3. After Ubuntu boots (30-60 seconds):")
    print("   ssh -p 2222 ubuntu@localhost")
    print()
    print("4. In VM, deploy FortiOS 8 lab")
    print()

print("Help & Documentation:")
print("  - Quick Reference: QUICK_REFERENCE.txt")
print("  - Setup Guide: VBOX_LINUX_GUIDE.md")
print("  - README: README.md")
print()
print("Support:")
print("  - Check troubleshooting in VBOX_LINUX_GUIDE.md")
print("  - Review logs: tail -f ~/.config/VirtualBox/VBoxSVC.log")
print()
print("="*70)
print("Happy hacking! 🎯")
print("="*70)